# Phase 1 — Data Preparation
## LAPD Crime Data 2020–2024

**Objective:** Transform the raw 1M-row CSV into a clean, enriched Parquet file ready for EDA and ML.

**What this notebook does:**
1. Load raw data and profile quality issues
2. Parse dates and engineer temporal features
3. Clean geographic coordinates
4. Decode and enrich victim demographics
5. Classify crimes into broad categories
6. Normalize premises and weapon fields
7. Add case resolution flags
8. Optimize memory via efficient dtypes
9. Export to Parquet

**Output:** `data/processed/lapd_clean.parquet` — the single source of truth for all downstream work.

---
## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Add src/ to path so we can import our cleaning module
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from src.prepare_data import (
    RAW_CSV, OUT_DIR,
    load_raw, parse_dates, engineer_temporal, clean_geo,
    clean_victim, classify_crimes, clean_premises_weapon,
    add_status_flags, parse_mocodes, optimize_types,
    clean, main
)

# Plot style
plt.style.use('dark_background')
sns.set_palette('tab10')
FIGDIR = ROOT / 'outputs' / 'figures'

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Setup complete. ROOT:', ROOT)

---
## 1. Load Raw Data & Initial Profile

Before touching anything, we need to understand what we are working with:
- Shape (rows × columns)
- Memory footprint
- Null counts per field
- Duplicate check on the primary key (`DR_NO`)

> **Rule:** Never clean data you haven't profiled. Cleaning decisions must be justified by evidence.

In [ ]:
print('Loading raw CSV...')
df_raw = load_raw(RAW_CSV)
raw_mem = df_raw.memory_usage(deep=True).sum() / 1e6

print(f'Shape  : {df_raw.shape[0]:>10,} rows × {df_raw.shape[1]} columns')
print(f'Memory : {raw_mem:>10.1f} MB')
df_raw.head(3)

In [ ]:
# Null profile
null_profile = pd.DataFrame({
    'dtype':     df_raw.dtypes,
    'nulls':     df_raw.isnull().sum(),
    'null_pct':  (df_raw.isnull().sum() / len(df_raw) * 100).round(1),
    'unique':    df_raw.nunique(),
}).sort_values('null_pct', ascending=False)

print('=== NULL PROFILE ===')
print(null_profile.to_string())

In [ ]:
# Visualize null rates
fig, ax = plt.subplots(figsize=(10, 7))
cols_with_nulls = null_profile[null_profile['null_pct'] > 0]

colors = ['#e05252' if p > 50 else '#e0883a' if p > 10 else '#4f8ef7'
          for p in cols_with_nulls['null_pct']]

bars = ax.barh(cols_with_nulls.index, cols_with_nulls['null_pct'], color=colors)
ax.set_xlabel('Null %', fontsize=11)
ax.set_title('Null Rates per Column — Raw Data', fontsize=13, fontweight='bold', pad=12)
ax.axvline(50, color='white', linestyle='--', alpha=0.3, linewidth=0.8)

for bar, val in zip(bars, cols_with_nulls['null_pct']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_null_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: High null rates in Weapon (67%), Cross Street (85%), Crm Cd 2-4 are EXPECTED — not data errors.')

In [ ]:
# Duplicate check on primary key
dupes = df_raw['DR_NO'].duplicated().sum()
print(f'Duplicate DR_NO entries: {dupes}')
if dupes > 0:
    print(df_raw[df_raw['DR_NO'].duplicated(keep=False)].sort_values('DR_NO').head(10))

---
## 2. Date & Time Engineering

**Problem:** Dates are stored as strings (`'04/11/2021 12:00:00 AM'`), and `TIME OCC` is a 4-digit integer (`1845` = 18:45).

**Solution:** Parse to proper datetime types and extract a full set of temporal features.

**New columns:**

| Column | Type | Description |
|---|---|---|
| `date_occ` | datetime | Parsed occurrence date |
| `date_rptd` | datetime | Parsed report date |
| `year`, `month`, `quarter` | int | Calendar periods |
| `week_of_year` | int | ISO week number |
| `day_of_week` | int | 0=Monday, 6=Sunday |
| `day_name` | str | 'Monday', 'Tuesday'… |
| `is_weekend` | bool | Saturday or Sunday |
| `hour` | int | 0–23 extracted from TIME OCC |
| `time_of_day` | category | Late Night / Morning / Afternoon / Evening |
| `season` | category | Winter / Spring / Summer / Fall |
| `days_to_report` | int | Lag between occurrence and report (days) |
| `reported_same_day` | bool | Lag == 0 |

In [ ]:
df = load_raw(RAW_CSV)
df = parse_dates(df)
df = engineer_temporal(df)

print('Date range (date_occ):', df['date_occ'].min(), '→', df['date_occ'].max())
print('Temporal features added:')
print([c for c in df.columns if c not in load_raw(RAW_CSV).columns][:15])

In [ ]:
# Reporting lag distribution
lag = df['days_to_report'].dropna()
lag_capped = lag.clip(upper=365)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(lag_capped, bins=100, color='#4f8ef7', edgecolor='none')
axes[0].set_title('Reporting Lag Distribution (days, capped at 365)', fontweight='bold')
axes[0].set_xlabel('Days between occurrence and report')
axes[0].set_ylabel('Incidents')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

same_day_pct = (lag == 0).mean() * 100
pcts = [same_day_pct, 100 - same_day_pct]
axes[1].pie(pcts, labels=['Same day', 'Later'], autopct='%1.1f%%',
            colors=['#3ecf8e', '#4f8ef7'], startangle=90)
axes[1].set_title('Reported Same Day vs. Later', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_reporting_lag.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Same-day reports: {same_day_pct:.1f}%')
print(f'Median lag: {lag.median():.0f} days')
print(f'Mean lag:   {lag.mean():.1f} days')
print(f'Max lag:    {lag.max():.0f} days')

---
## 3. Geographic Cleaning

**Problem:** Some records have `LAT=0, LON=0` ("Null Island") — geocoding failures in the LAPD system.

**Solution:**
- Flag records outside the City of LA bounding box (`LAT` 33.7–34.4, `LON` −118.7 to −117.9)
- Set invalid coordinates to `NaN` (don't drop — we keep the crime record, just can't map it)
- Strip excessive whitespace from address strings (fixed-width RMS export artifact)

In [ ]:
df = clean_geo(df)

invalid = (~df['valid_geo']).sum()
total   = len(df)
print(f'Invalid coordinates: {invalid:,} ({invalid/total*100:.3f}%)')
print(f'Valid coordinates:   {total-invalid:,} ({(total-invalid)/total*100:.3f}%)')

In [ ]:
# Scatter of valid coordinates to verify they fall within LA
sample = df[df['valid_geo']].sample(20_000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(sample['LON'], sample['LAT'], s=0.3, alpha=0.3, color='#4f8ef7')
ax.set_title('20k Random Crime Locations (valid coords only)', fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(FIGDIR / 'p1_geo_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Victim Demographics

**Problems:**
- `Vict Age == 0` is LAPD's placeholder for "unknown" — not literal age 0
- `Vict Sex` and `Vict Descent` use single-letter codes undocumented in the raw data

**Solution:** Decode all codes to readable labels. Add age group buckets and a broader descent grouping for high-level analysis.

**New columns:** `vict_age`, `age_group`, `vict_sex`, `vict_descent`, `descent_group`

In [ ]:
df = clean_victim(df)

# Check decoding
print('=== VICT SEX (decoded) ===')
print(df['vict_sex'].value_counts())
print()
print('=== DESCENT GROUP ===')
print(df['descent_group'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Age distribution (non-null, non-zero)
ages = df['vict_age'].dropna()
ages = ages[ages.between(1, 99)]
axes[0].hist(ages, bins=50, color='#7c5cbf', edgecolor='none')
axes[0].set_title('Victim Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Incidents')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Sex
sex_counts = df['vict_sex'].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values,
            color=['#4f8ef7', '#e05252', '#888'])
axes[1].set_title('Victim Sex', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Descent group
descent = df[df['descent_group'] != 'Unknown']['descent_group'].value_counts()
colors_d = ['#4f8ef7','#3ecf8e','#e05252','#e0883a','#7c5cbf','#e0c066']
axes[2].barh(descent.index, descent.values, color=colors_d)
axes[2].set_title('Victim Descent Group\n(excl. Unknown)', fontweight='bold')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_victim_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Crime Classification

The raw `Crm Cd Desc` has ~140 distinct crime descriptions — too granular for dashboards and ML features.

We map them to **14 broad categories** using keyword matching:

| Category | Examples |
|---|---|
| Violent – Homicide | Homicide, Manslaughter |
| Violent – Sexual Assault | Rape, Sexual Penetration |
| Violent – Robbery | Robbery |
| Violent – Aggravated Assault | ADW, Aggravated Assault |
| Violent – Assault & Battery | Simple Assault, Battery |
| Domestic Violence | Intimate Partner offenses |
| Property – Burglary | Burglary (residential/commercial) |
| Property – Theft | Petty/Grand Theft, Shoplifting |
| Property – Vandalism | Vandalism (felony/misdemeanor) |
| Vehicle Crime | Stolen vehicle, Theft from vehicle |
| Identity / Fraud | Identity Theft, Fraud, Forgery |
| Sex Offense | Lewd acts, Peeping Tom |
| Drug Offense | Narcotics |
| Other | Everything else |

We also add `is_violent` (bool), `is_property` (bool), and `part_label` (UCR class).

In [ ]:
df = classify_crimes(df)

print('=== CRIME CATEGORY DISTRIBUTION ===')
cat_counts = df['crime_category'].value_counts()
for cat, n in cat_counts.items():
    bar = '█' * int(n / 5000)
    print(f'{cat:<40} {n:>7,}  {bar}')

print(f'\nViolent crimes:  {df["is_violent"].sum():>7,} ({df["is_violent"].mean()*100:.1f}%)')
print(f'Property crimes: {df["is_property"].sum():>7,} ({df["is_property"].mean()*100:.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

cat_counts_sorted = cat_counts.sort_values()
colors_cat = ['#e05252' if 'Violent' in c or c in ('Domestic Violence','Sex Offense','Human Trafficking','Crimes Against Children')
               else '#4f8ef7' if 'Property' in c or c == 'Vehicle Crime'
               else '#e0883a'
               for c in cat_counts_sorted.index]

bars = ax.barh(cat_counts_sorted.index, cat_counts_sorted.values, color=colors_cat)
ax.set_xlabel('Number of Incidents', fontsize=11)
ax.set_title('Crime Categories — LAPD 2020–2024\n(red=violent, blue=property, orange=other)',
             fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

for bar, val in zip(bars, cat_counts_sorted.values):
    ax.text(val + 1000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_crime_categories.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Premises & Weapon Normalization

**Premises:** ~298 distinct premise types → 5 groups: Residential / Commercial / Vehicle / Public / Other

**Weapon:** 67% of incidents have no weapon (expected). The remaining 33% map to:
Firearm / Knife / Physical Force / Blunt Object / Unknown-Other

In [ ]:
df = clean_premises_weapon(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Premises group
prem = df['premises_group'].value_counts()
axes[0].bar(prem.index, prem.values,
            color=['#4f8ef7','#3ecf8e','#e0883a','#7c5cbf','#888'])
axes[0].set_title('Premises Group', fontweight='bold')
axes[0].set_xlabel('Premises Type')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
axes[0].tick_params(axis='x', rotation=20)

# Weapon category
weap = df['weapon_category'].value_counts()
axes[1].bar(weap.index, weap.values,
            color=['#888','#e05252','#e0883a','#4f8ef7','#7c5cbf'])
axes[1].set_title('Weapon Category', fontweight='bold')
axes[1].set_xlabel('Weapon Type')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_premises_weapon.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Case Status Flags

**Clearance rate** = percentage of cases resolved (arrest or other disposition).

Cleared statuses: `AA` (Adult Arrest), `AO` (Adult Other), `JA` (Juv Arrest), `JO` (Juv Other)

New columns: `cleared` (bool), `arrested` (bool)

In [ ]:
df = add_status_flags(df)

overall_cleared  = df['cleared'].mean()  * 100
overall_arrested = df['arrested'].mean() * 100
print(f'Overall clearance rate : {overall_cleared:.1f}%')
print(f'Overall arrest rate    : {overall_arrested:.1f}%')

# Clearance by crime category
clearance_by_cat = (
    df.groupby('crime_category', observed=True)['cleared']
    .mean()
    .sort_values(ascending=True)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors_clr = ['#3ecf8e' if v >= 30 else '#e0883a' if v >= 15 else '#e05252'
              for v in clearance_by_cat.values]
ax.barh(clearance_by_cat.index, clearance_by_cat.values, color=colors_clr)
ax.axvline(overall_cleared, color='white', linestyle='--', linewidth=1, alpha=0.6,
           label=f'Overall avg: {overall_cleared:.1f}%')
ax.set_xlabel('Clearance Rate (%)', fontsize=11)
ax.set_title('Case Clearance Rate by Crime Category', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=10)

for i, (cat, val) in enumerate(clearance_by_cat.items()):
    ax.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGDIR / 'p1_clearance_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Mocodes Parsing

`Mocodes` contains space-separated codes (e.g., `'0416 0334 2004'`). We split into a list and count how many M.O. codes each incident has — more codes generally means a more complex crime report.

In [ ]:
df = parse_mocodes(df)

print('Mocode count distribution:')
print(df['mocode_count'].value_counts().sort_index().head(15).to_string())
print(f'\nMax mocodes in one incident: {df["mocode_count"].max()}')
print(f'Avg mocodes per incident:    {df["mocode_count"].mean():.2f}')

---
## 9. Type Optimization

Low-cardinality string columns are converted to `category` dtype — this reduces memory dramatically and speeds up groupby operations.

We already loaded numeric columns with tight dtypes (`int8`, `int16`, `float32`) instead of the default `int64`/`float64`.

In [ ]:
mem_before = df.memory_usage(deep=True).sum() / 1e6
df = optimize_types(df)
mem_after = df.memory_usage(deep=True).sum() / 1e6

print(f'Memory before optimization : {mem_before:.1f} MB')
print(f'Memory after  optimization : {mem_after:.1f} MB')
print(f'Reduction                  : {(mem_before-mem_after)/mem_before*100:.1f}%')

---
## 10. Final Quality Check

Before exporting, we do one final pass to confirm:
- No unexpected nulls in columns that should be complete
- All new derived columns are present
- Shape is as expected

In [ ]:
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory:      {df.memory_usage(deep=True).sum()/1e6:.1f} MB\n')

# New columns added by this pipeline
original_cols = set(load_raw(RAW_CSV).columns)
new_cols      = [c for c in df.columns if c not in original_cols]
print(f'New columns added ({len(new_cols)}):')
for col in new_cols:
    null_pct = df[col].isnull().mean() * 100
    print(f'  {col:<25} dtype={str(df[col].dtype):<15} nulls={null_pct:.1f}%')

In [ ]:
# Final sample
new_col_sample = [
    'date_occ', 'year', 'month', 'hour', 'time_of_day', 'season', 'is_weekend',
    'days_to_report', 'valid_geo', 'vict_age', 'age_group', 'vict_sex',
    'vict_descent', 'descent_group', 'crime_category', 'is_violent',
    'premises_group', 'weapon_category', 'cleared', 'arrested', 'mocode_count'
]
df[new_col_sample].head(5)

---
## 11. Export to Parquet

We use **Parquet with Snappy compression** because:
- 5–10× smaller than CSV for this kind of data
- Column-oriented: analytical queries only read the columns they need
- Preserves dtypes (no re-parsing on load)
- Standard format for pandas, Spark, DuckDB, and most BI tools

We also export a **5% sample** for rapid Power BI prototyping.

In [ ]:
# Full dataset
out_full = OUT_DIR / 'lapd_clean.parquet'
df.to_parquet(out_full, index=False, engine='pyarrow', compression='snappy')
print(f'Full parquet  : {out_full}')
print(f'File size     : {out_full.stat().st_size / 1e6:.1f} MB')

# 5% sample
out_sample = OUT_DIR / 'lapd_sample_5pct.parquet'
df.sample(frac=0.05, random_state=42).to_parquet(out_sample, index=False, engine='pyarrow')
print(f'\n5% sample     : {out_sample}')
print(f'File size     : {out_sample.stat().st_size / 1e6:.1f} MB')

print('\nPhase 1 complete. Ready for Phase 2 — EDA.')

---
## Summary

| Step | Action | New columns |
|---|---|---|
| Dates | Parsed 2 date strings to datetime | `date_occ`, `date_rptd` |
| Temporal | Extracted calendar + time features | 12 columns |
| Geography | Flagged invalid coords, stripped addresses | `valid_geo` |
| Victim | Decoded codes, added age groups | 5 columns |
| Crime | Mapped 140 codes → 14 categories | `crime_category`, `is_violent`, `is_property`, `part_label` |
| Premises | Grouped 298 types → 5 groups | `premises_group` |
| Weapon | Grouped weapons, coded nulls as "No Weapon" | `weapon_category` |
| Status | Added clearance + arrest flags | `cleared`, `arrested` |
| Mocodes | Split multi-value field into list | `mocode_list`, `mocode_count` |
| Types | Converted strings to `category` | (in-place) |

**Output:** `data/processed/lapd_clean.parquet` — `1,004,894` rows × `~49` columns